# 04b — Ensemble residual + LightGBM: OD da estação EF01 (CETESB)

**Objetivo:** testar se o ensemble com piso no sazonal-naive bate a régua do OD (sazonal-naive 0,1525 rolante / 0,1550 holdout), no mesmo desenho do 00b/01b/02b/03b (avaliação em janelas L=8640/H=288, split 70/15/15 + holdout de 10 dias, 10 origens diárias, segmento limpo 01/06 → 21/07).
**Modelos:** (i) **lgbm** — LightGBM direto, um regressor por passo do horizonte (288), features de lags sazonais + médias móveis + hora-do-dia; (ii) **dlres** — DLinear-5min treinado no **resíduo** `r = Y − snaive(X)`; (iii) **lstnet (02b)** recarregado (só inferência); (iv) **ens** — combinação NNLS dos quatro (sazonal + 3) com pesos fitados na val.
**Dados:** `dados/ef01-mogi-das-cruzes_oxigenio-dissolvido_2026-06-01_a_2026-08-31.csv` — ver `dados/README.md`.


In [ ]:
import json
import random
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

ROOT = Path.cwd() if (Path.cwd() / "dados").exists() else Path.cwd().parent
CSV = ROOT / "dados" / "ef01-mogi-das-cruzes_oxigenio-dissolvido_2026-06-01_a_2026-08-31.csv"
OUT = ROOT / "resultados" / "04b-ensemble-od"
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(parents=True, exist_ok=True)

# --- protocolo travado (igual ao 00) ---
L, H = 8640, 288
SEASON = 288
INTERP_LIMIT = 24
SEG_FIM = "2026-07-21 01:05"  # fim do segmento limpo (início do gap de 16,4 dias)
HOLDOUT_DIAS = 10
# --- 04b: LightGBM direto + residual + ensemble (piso = sazonal-naive) ---
LN = 2016                  # contexto p/ residual-DLinear (últimos passos de X)
LGB_LAGS = [288, 576, 2016]
LGB_EST, LGB_LR, LGB_LEAVES = 150, 0.05, 31
LGB_STRIDE = 2             # treino do lgbm
DL_EPOCHS, DL_PAT = 30, 5
TRAIN_STRIDE, ENS_STRIDE = 4, 4
SEED = 42


random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cpu")
print("ROOT:", ROOT, "| CSV existe:", CSV.exists(), "| torch:", torch.__version__)



## 1. Carga
Formato CETESB: `;`, decimal com vírgula, `windows-1252`, linha 1 = validação, linha 2 = cabeçalho.


In [ ]:
df = pd.read_csv(CSV, sep=";", decimal=",", encoding="windows-1252",
                 skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
colvar = [c for c in df.columns if c != "Data hora"][0]
df = df.rename(columns={"Data hora": "ds", colvar: "y"}).sort_values("ds").reset_index(drop=True)
print(colvar, "|", df.shape, df["ds"].min(), "→", df["ds"].max())
print("faltantes:", int(df['y'].isna().sum()), f"({100*df['y'].isna().mean():.1f}%)")
df.describe()



## 2. EDA — perfil, o gap de 16 dias e ciclo diário


In [ ]:
isna = df["y"].isna().to_numpy()
bounds = np.where(np.diff(np.concatenate([[False], isna, [False]])))[0]
runs = sorted([(bounds[i], bounds[i+1]-1) for i in range(0, len(bounds), 2)],
              key=lambda r: r[1]-r[0], reverse=True)
print("top 5 gaps:")
for a, b in runs[:5]:
    print(f"  {df.ds[a]} → {df.ds[b]}  ({(b-a+1)*5/60:.1f} h)")

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
ax[0].plot(df["ds"], df["y"], lw=0.4)
ax[0].axvspan(pd.Timestamp("2026-07-21 01:10"), pd.Timestamp("2026-08-06 11:30"),
              color="r", alpha=0.2, label="sensor morto (16,4 dias)")
ax[0].set_title("OD EF01 — série completa (faixa vermelha = gap, fora do experimento)")
ax[0].set_ylabel("OD (mg/L)")
ax[0].legend(fontsize=8)
df["y"].hist(bins=60, ax=ax[1])
ax[1].set_title("Distribuição do OD")
df.assign(hora=df["ds"].dt.hour).boxplot(column="y", by="hora", ax=ax[2], grid=False)
ax[2].set_title("OD por hora do dia (ciclo diário?)")
ax[2].set_xlabel("hora")
fig.tight_layout()
fig.savefig(OUT / "figs" / "01-eda.png")
print("fig salva:", OUT / "figs" / "01-eda.png")



## 3. Limpeza + recorte do segmento limpo
Grade de 5 min, interpolação máx. 2 h e **corte em 21/07 01:05** (antes do gap). Tudo a jusante usa só o segmento 01/06 → 21/07.


In [ ]:
idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
s_full = df.set_index("ds")["y"].reindex(idx)
s = s_full.loc[:SEG_FIM].interpolate(method="time", limit=INTERP_LIMIT)
print(f"segmento: {s.index.min()} → {s.index.max()} ({len(s)} slots = {len(s)*5/60/24:.1f} dias)")
print(f"NaN após interpolação (limite {INTERP_LIMIT}): {int(s.isna().sum())}")

amostra = slice("2026-06-08", "2026-06-15")
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(s_full[amostra].index, s_full[amostra].values, ".", ms=2, label="cru (com faltantes)")
ax.plot(s[amostra].index, s[amostra].values, lw=0.8, label=f"interpolado (limite {INTERP_LIMIT})")
ax.legend(); ax.set_title("Exemplo de preenchimento — semana 08–15/06")
fig.tight_layout(); fig.savefig(OUT / "figs" / "02-limpeza.png")
print("fig salva")



## 4. Estacionariedade (ADF) e decomposição STL
Idêntico ao 00 (últimos 4032 pontos do treino, período 288).


In [ ]:
n_total = len(s)
n_train = int(n_total * 0.70)
train = s.iloc[:n_train].dropna()
stat, pval, *_ = adfuller(train.values)
print(f"ADF stat={stat:.2f} p-valor={pval:.3g} → {'estacionária' if pval < 0.05 else 'NÃO estacionária'}")

stl = STL(train.iloc[-4032:], period=SEASON, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 6)
fig.savefig(OUT / "figs" / "03-stl.png")
print("fig salva")



## 5. Janelamento + holdout puro
Amostras `(L=8640 → H=288)` por janela deslizante, só janelas 100% observadas. Pré-holdout: split 70/15/15 **sem shuffle**. Holdout: últimos 10 dias + 10 origens diárias. **Idêntico ao 00** — o ensemble será avaliado nestas mesmas janelas.


In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

v = s.to_numpy()
W = sliding_window_view(v, L + H)
ok = ~np.isnan(W).any(axis=1)
W = W[ok]
X, Y = W[:, :L], W[:, L:]
ends = s.index[L + H - 1:][ok]
n = len(X)
ZONE = s.index.max() - pd.Timedelta(days=HOLDOUT_DIAS)
is_hold = ends >= (ZONE + pd.Timedelta(minutes=5 * (H - 1)))
ho = np.where(is_hold)[0]
pre = np.where(~is_hold)[0]
i1, i2 = int(len(pre) * 0.70), int(len(pre) * 0.85)
tr, va, te = pre[:i1], pre[i1:i2], pre[i2:]
splits = {"train": tr, "val": va, "test": te, "holdout": ho}
for k, idx in splits.items():
    print(f"{k}: {len(idx)} janelas | alvos {ends[idx[0]].date()} → {ends[idx[-1]].date()}")
print(f"janelas descartadas (com NaN): {len(s) - L - H + 1 - n}")
print(f"zona holdout (alvos): {ZONE.date()} → {s.index.max().date()}")
daily_ends = [ZONE + pd.Timedelta(minutes=5 * (H - 1 + H * k)) for k in range(HOLDOUT_DIAS)]
daily_idx = np.array([int(np.where(ends == d)[0][0]) for d in daily_ends])
print("dias previstos:", [str(ends[i].date()) for i in daily_idx])
TR_END = ends[tr[-1]]



## 6. Baselines baratos (teste rolante + holdout)
Persistência, sazonal-naive (lag 288) e média móvel 288 — vetorizados, mesmos do 00.


In [ ]:
def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
def mape(a, b, eps=1e-6): return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), eps))) * 100)
def smape(a, b, eps=1e-6): return float(np.mean(2*np.abs(a-b) / (np.abs(a)+np.abs(b)+eps)) * 100)

def metricas(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred), "sMAPE": smape(y_true, y_pred)}

def cheap_preds(X_):
    return {
        "persistencia": np.repeat(X_[:, -1:], H, axis=1),
        "sazonal_naive_288": np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1),
        "media_movel_288": np.repeat(X_[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
    }

Xte, Yte = X[te], Y[te]
Xho, Yho = X[ho], Y[ho]
pred_te = cheap_preds(Xte)
pred_ho = cheap_preds(Xho)
print("teste rolante:")
print(pd.DataFrame({m: metricas(Yte, p) for m, p in pred_te.items()}).T.round(4).to_string())



## 7. LightGBM direto no resíduo (um modelo por passo do horizonte)
Alvo = resíduo `r = Y − snaive(X)` (piso = sazonal-naive; árvores aprendem só a correção). Features por origem (só passado): lags recentes (1–144) + sazonais (287/288/289/576/2016), média/desvio da mesma fase nos últimos 7 dias, médias/desvios móveis (12/36/144/288/2016) + hora-do-dia do passo-alvo. 288 `LGBMRegressor` (150 árvores) — sem rollout, sem vazamento. Modelos salvos em `modelos/lgbm_steps.pkl`.


In [ ]:
import lightgbm as lgb

def base_feats(Xb, E):
    cols = [Xb[:, -k] for k in [1, 2, 3, 6, 12, 24, 36, 72, 144, 287, 288, 289, 576, 2016]]
    phase = np.stack([Xb[:, L - 288*k] for k in range(1, 8)], axis=1)
    cols += [phase.mean(1), phase.std(1)]
    for w in [12, 36, 144, 288]:
        cols += [Xb[:, -w:].mean(1), Xb[:, -w:].std(1)]
    cols += [Xb[:, -2016:].mean(1)]
    F = np.stack(cols, axis=1)
    em = (E.hour.to_numpy()*60 + E.minute.to_numpy()).astype(np.float32)
    return F.astype(np.float32), em

def hour_sincos(em, j):
    hh = ((em - (H - 1 - j)*5) % 1440 // 60).astype(np.float32)
    return np.sin(2*np.pi*hh/24).astype(np.float32), np.cos(2*np.pi*hh/24).astype(np.float32)

tr2 = tr[::LGB_STRIDE]
Xb_tr = X[tr2]  # hoist: X[tr2] avaliado 1x (0,36 GB); sem isso a compreensão abaixo retinha 288 cópias (~100 GB, OOM)
Ftr, emtr = base_feats(Xb_tr, ends[tr2])
Str = np.stack([Xb_tr[:, L - SEASON + h] for h in range(H)], axis=1)  # base sazonal (piso)
Rtr = (Y[tr2] - Str).astype(np.float32)  # resíduo: o que as árvores aprendem
print(f"features: {Ftr.shape} + hora do passo | resíduo std: {Rtr.std():.4f}")

models = []
t0 = time.time()
for j in range(H):
    sh, ch = hour_sincos(emtr, j)
    m = lgb.LGBMRegressor(n_estimators=LGB_EST, learning_rate=LGB_LR, num_leaves=LGB_LEAVES,
                          verbosity=-1, force_col_wise=True)
    m.fit(np.column_stack([Ftr, sh, ch]), Rtr[:, j])
    models.append(m)
    if (j + 1) % 72 == 0:
        print(f"  lgbm {j+1}/{H} ...", flush=True)
print(f"lgbm: {len(models)} modelos em {time.time()-t0:.0f}s")
with open(OUT / "modelos" / "lgbm_steps.pkl", "wb") as f:
    pickle.dump(models, f)
print("modelos salvos: modelos/lgbm_steps.pkl")

def prevê_lgbm(idxs):
    ii = np.asarray(idxs)
    Xb = X[ii]  # hoist: avaliado 1x; sem isso a compreensão abaixo retinha 288 cópias (OOM)
    F, em = base_feats(Xb, ends[ii])
    S = np.stack([Xb[:, L - SEASON + h] for h in range(H)], axis=1)
    P = np.empty((len(ii), H), dtype=np.float32)
    for j, m in enumerate(models):
        sh, ch = hour_sincos(em, j)
        P[:, j] = S[:, j] + m.predict(np.column_stack([F, sh, ch]))
    return P

imp = np.mean([m.booster_.feature_importance(importance_type="gain") for m in models], axis=0)
nomes = ["lag1", "lag2", "lag3", "lag6", "lag12", "lag24", "lag36", "lag72", "lag144",
         "lag287", "lag288", "lag289", "lag576", "lag2016", "seasmean7", "seasstd7",
         "rm12", "rs12", "rm36", "rs36", "rm144", "rs144", "rm288", "rs288", "rm2016",
         "hora_sin", "hora_cos"]
ordem = np.argsort(imp)[::-1]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh([nomes[k] for k in ordem], imp[ordem])
ax.set_title("LightGBM — importância média das features (gain, 288 modelos)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "07-importancia-lgbm.png")
print("top features:", [(nomes[k], round(float(imp[k]), 1)) for k in ordem[:5]])
print("fig salva: 07-importancia-lgbm.png")



## 8. Residual-DLinear + régua LSTNet + ensemble NNLS
DLinear-5min treinado no resíduo `r = Y − snaive(X)` (piso = sazonal-naive). Régua do 02 por checkpoint. Pesos do ensemble via NNLS nas janelas da val. Inferência de tudo em todas as origens do teste, holdout e holdout diário.


In [ ]:
def snaive(X_):
    return np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1)

class DLinearLite(nn.Module):
    def __init__(self, k=25):
        super().__init__()
        self.pool = nn.AvgPool1d(k, stride=1, padding=k // 2)
        self.lin_t = nn.Linear(LN, H)
        self.lin_s = nn.Linear(LN, H)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        mu = x.mean(dim=1, keepdim=True); sg = x.std(dim=1, keepdim=True).clamp_min(1e-3)
        xn = self.gamma * (x - mu) / sg + self.beta
        t = self.pool(xn.unsqueeze(1)).squeeze(1)
        y = self.lin_t(t) + self.lin_s(xn - t)
        return (y - self.beta) / self.gamma.clamp_min(1e-3) * sg  # sem +mu: alvo é resíduo de média ~0

def monta_res(idxs):
    ii = np.asarray(idxs)
    return X[ii][:, -LN:].astype(np.float32), (Y[ii] - snaive(X[ii])).astype(np.float32)

Xr_tr, Rr_tr = monta_res(tr[::TRAIN_STRIDE])
Xr_va, Rr_va = monta_res(va[::TRAIN_STRIDE])
print(f"residual: treino {Xr_tr.shape} val {Xr_va.shape}")
dlres = DLinearLite().to(DEVICE)
opt = torch.optim.Adam(dlres.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
tr_loader = DataLoader(TensorDataset(torch.from_numpy(Xr_tr), torch.from_numpy(Rr_tr)), batch_size=512, shuffle=True)
va_loader = DataLoader(TensorDataset(torch.from_numpy(Xr_va), torch.from_numpy(Rr_va)), batch_size=512)
best, patience = float("inf"), 0
t0 = time.time()
for ep in range(1, DL_EPOCHS + 1):
    dlres.train()
    for xb, yb in tr_loader:
        opt.zero_grad(); loss = loss_fn(dlres(xb), yb); loss.backward(); opt.step()
    dlres.eval(); vl = 0.0
    with torch.no_grad():
        for xb, yb in va_loader:
            vl += float(loss_fn(dlres(xb), yb)) * len(xb)
    vl /= len(va_loader.dataset)
    tag = ""
    if vl < best:
        best, patience = vl, 0
        torch.save({"state": dlres.state_dict()}, OUT / "modelos" / "dlinear_res_od.pt")
        tag = " *"
    else:
        patience += 1
    print(f"dlres ep {ep:02d} val={vl:.5f}{tag}", flush=True)
    if patience >= DL_PAT:
        break
print(f"dlres em {time.time()-t0:.0f}s | melhor val={best:.5f}")
dlres.load_state_dict(torch.load(OUT / "modelos" / "dlinear_res_od.pt", map_location="cpu", weights_only=False)["state"])
dlres.eval()

# régua 02 (LSTNet, só inferência)
class LSTNet1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv1d(3, 32, kernel_size=12, stride=6)
        self.gru = nn.GRU(32, 64, batch_first=True)
        self.skipcell = nn.GRUCell(32, 32)
        self.head = nn.Linear(96, 288)
        self.ar = nn.Linear(288, 288)
        self.drop = nn.Dropout(0.1)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, xv, tod):
        mu = xv.mean(dim=1, keepdim=True); sg = xv.std(dim=1, keepdim=True).clamp_min(1e-3)
        vn = self.gamma * (xv - mu) / sg + self.beta
        f = self.drop(torch.relu(self.conv(torch.cat([vn.unsqueeze(1), tod.transpose(1, 2)], dim=1))))
        f = f.transpose(1, 2)
        _, h = self.gru(f)
        B, T, _ = f.shape
        hs = torch.zeros(B, 32, device=f.device)
        states = [hs]
        for t in range(T):
            prev = states[t - 48] if t - 48 >= 0 else states[0]
            hs = self.skipcell(f[:, t, :], prev)
            states.append(hs)
        g = self.gamma.clamp_min(1e-3)
        yn = self.head(self.drop(torch.cat([h.squeeze(0), hs], dim=1)))
        ya = self.ar(vn[:, -288:])
        return (yn + ya - self.beta) / g * sg + mu

ckpt02 = torch.load(ROOT / "resultados" / "02b-lstnet-od" / "modelos" / "lstnet_od.pt",
                    map_location="cpu", weights_only=False)
ruler = LSTNet1D().to(DEVICE)
ruler.load_state_dict(ckpt02["state"])
ruler.eval()
SIN5 = np.sin(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
COS5 = np.cos(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
Tln = sliding_window_view(np.stack([SIN5, COS5], axis=1), 2016, axis=0).transpose(0, 2, 1).astype(np.float32)
pos_end = s.index.get_indexer(ends - pd.Timedelta(minutes=5*H))
rowln = pos_end - 2016 + 1

@torch.no_grad()
def prevê_tudo(idxs, batch=256):
    ii = np.asarray(idxs)
    Ps = snaive(X[ii])
    Gb = prevê_lgbm(ii)
    outs = []
    Xt = torch.from_numpy(X[ii][:, -2016:].astype(np.float32))
    for b in range(0, len(Xt), batch):
        outs.append(dlres(Xt[b:b+batch]).numpy())
    Dr = Ps + np.concatenate(outs)
    return Ps, Gb, Dr

# inferência da régua (precisa do contexto 2016 + ToD)
Wln2 = sliding_window_view(s.to_numpy().astype(np.float32), 2016)
@torch.no_grad()
def prevê_ruler(idxs, batch=256):
    ii = np.asarray(idxs)
    outs = []
    for b in range(0, len(ii), batch):
        xb = torch.from_numpy(Wln2[rowln[ii[b:b+batch]]])
        tb = torch.from_numpy(Tln[rowln[ii[b:b+batch]]])
        outs.append(ruler(xb, tb).numpy())
    return np.concatenate(outs)

# pesos NNLS na val
from scipy.optimize import nnls
va2 = va[::ENS_STRIDE]
Ps_v, Gb_v, Dr_v = prevê_tudo(va2)
Pn_v = prevê_ruler(va2)
A = np.column_stack([Ps_v.ravel(), Pn_v.ravel(), Gb_v.ravel(), Dr_v.ravel()])
w, _ = nnls(A, Y[va2].ravel())
pesos = {k: round(float(v), 4) for k, v in zip(["sazonal", "lstnet", "lgbm", "dlres"], w)}
json.dump({"pesos": pesos, "mode": "nnls-ensemble sobre sazonal+lstnet+lgbm+dlres"},
          open(OUT / "modelos" / "ensemble.json", "w"))
json.dump({"mode": "lgbm-direct-288 + dlinear-residual + nnls-ensemble", "LN": LN},
          open(OUT / "modelos" / "normalizacao.json", "w"))
print("pesos ensemble (nnls na val):", pesos)

def ensemble(Ps, Pn, Gb, Dr):
    return w[0]*Ps + w[1]*Pn + w[2]*Gb + w[3]*Dr

t0 = time.time()
Ps_te, Gb_te, Dr_te = prevê_tudo(te); Pn_te = prevê_ruler(te)
En_te = ensemble(Ps_te, Pn_te, Gb_te, Dr_te)
Ps_ho, Gb_ho, Dr_ho = prevê_tudo(ho); Pn_ho = prevê_ruler(ho)
Ps_d, Gb_d, Dr_d = prevê_tudo(daily_idx); Pn_d = prevê_ruler(daily_idx)
En_d = ensemble(Ps_d, Pn_d, Gb_d, Dr_d)
print(f"inferência em {time.time()-t0:.0f}s")
print("LGBM teste:", {k: round(v, 4) for k, v in metricas(Yte, Gb_te).items()})
print("DLRES teste:", {k: round(v, 4) for k, v in metricas(Yte, Dr_te).items()})
print("ENS teste:", {k: round(v, 4) for k, v in metricas(Yte, En_te).items()})
print("ENS holdout:", {k: round(v, 4) for k, v in metricas(Y[daily_idx], En_d).items()})



## 9. Comparação final + holdout dia a dia
Tabela do teste rolante (todas as origens), tabela do holdout diário (10 dias) e MAE por dia. Réguas do 00b impressas para referência.


In [ ]:
linhas = {m: metricas(Yte, p) for m, p in pred_te.items()}
linhas["lstnet"] = metricas(Yte, Pn_te)
linhas["lgbm"] = metricas(Yte, Gb_te)
linhas["dlres"] = metricas(Yte, Dr_te)
linhas["ens"] = metricas(Yte, En_te)
tab = pd.DataFrame(linhas).T.round(4)
tab.to_csv(OUT / "metricas_baseline.csv")
print("=== teste rolante ===")
print(tab.to_string())

Yd = Y[daily_idx]
diario = {m: metricas(Yd, cheap_preds(X[daily_idx])[m]) for m in pred_te}
diario["lstnet"] = metricas(Yd, Pn_d)
diario["lgbm"] = metricas(Yd, Gb_d)
diario["dlres"] = metricas(Yd, Dr_d)
diario["ens"] = metricas(Yd, En_d)
tab_d = pd.DataFrame(diario).T.round(4)
tab_d.to_csv(OUT / "metricas_holdout.csv")
print("=== holdout diário (10 dias) ===")
print(tab_d.to_string())

por_dia = pd.DataFrame(
    {m: [mae(Yd[k:k+1], cheap_preds(X[daily_idx])[m][k:k+1]) for k in range(len(Yd))]
     for m in pred_te},
    index=[str(ends[i].date()) for i in daily_idx])
por_dia["lstnet"] = [mae(Yd[k:k+1], Pn_d[k:k+1]) for k in range(len(Yd))]
por_dia["lgbm"] = [mae(Yd[k:k+1], Gb_d[k:k+1]) for k in range(len(Yd))]
por_dia["dlres"] = [mae(Yd[k:k+1], Dr_d[k:k+1]) for k in range(len(Yd))]
por_dia["ens"] = [mae(Yd[k:k+1], En_d[k:k+1]) for k in range(len(Yd))]
print(por_dia.round(4).to_string())
print(f"\nRégua 02 (teste rolante): lstnet = 0.0456 | este exp: {tab['MAE'].idxmin()} = {tab['MAE'].min():.4f}")
print(f"Régua 02 (holdout diário): lstnet = 0.0446 | este exp: {tab_d['MAE'].idxmin()} = {tab_d['MAE'].min():.4f}")
print(f"pesos ensemble: {pesos}")



In [ ]:
E = ends[te]
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
for ax, k in zip(axes, [0, len(Xte)//2, -1]):
    tc = pd.date_range(E[k] - pd.Timedelta(minutes=5*(H+2015)), E[k] - pd.Timedelta(minutes=5*H), freq="5min")
    ax.plot(tc, Xte[k][-2016:], lw=0.8, label="contexto (cauda 7d)")
    tf = pd.date_range(E[k] - pd.Timedelta(minutes=5*(H-1)), E[k], freq="5min")
    ax.plot(tf, Yte[k], "k-", lw=1.5, label="real")
    ax.plot(tf, pred_te["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    ax.plot(tf, pred_te["persistencia"][k], ":", lw=1, label="persistência")
    ax.plot(tf, Pn_te[k], lw=1, alpha=0.6, label="lstnet(02)")
    ax.plot(tf, Gb_te[k], lw=1, alpha=0.9, label="lgbm")
    ax.plot(tf, En_te[k], lw=1.2, alpha=0.9, label="ens")
    ax.set_title(f"origem {E[k]}")
    ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "04-forecasts.png")

fig, ax = plt.subplots(figsize=(8, 4))
tab["MAE"].sort_values().plot.barh(ax=ax)
ax.set_title("MAE no teste rolante — baselines + LSTNet (menor = melhor)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "05-mae.png")

fig, axes = plt.subplots(5, 2, figsize=(14, 12), sharey=False)
for ax, k in zip(axes.ravel(), range(len(Yd))):
    tf = pd.date_range(ends[daily_idx[k]] - pd.Timedelta(minutes=5*(H-1)), ends[daily_idx[k]], freq="5min")
    ax.plot(tf, Yd[k], "k-", lw=1.2, label="real")
    ax.plot(tf, cheap_preds(X[daily_idx])["persistencia"][k], ":", lw=1, label="persistência")
    ax.plot(tf, cheap_preds(X[daily_idx])["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    ax.plot(tf, Pn_d[k], lw=1, alpha=0.6, label="lstnet(02)")
    ax.plot(tf, Gb_d[k], lw=1, alpha=0.9, label="lgbm")
    ax.plot(tf, En_d[k], lw=1.2, alpha=0.9, label="ens")
    ax.set_title(f"dia previsto {ends[daily_idx[k]].date()} (MAE ens={por_dia['ens'].iloc[k]:.3f} vs saz={por_dia['sazonal_naive_288'].iloc[k]:.3f})")
    ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "figs" / "06-holdout-dias.png")
print("figs salvas")



## 10. Conclusões e próximos passos

- A régua do 00b (sazonal-naive 0,1525 / 0,1550) está impressa na §9 e o LSTNet do 02b vem recarregado por checkpoint na mesma tabela.
- O ensemble tem piso no sazonal-naive: a correção residual só soma o que aprende; pesos NNLS fitados só na val.
- A importância das features do LightGBM (`07`) diz quais lags o modelo usa — no OD, a correção de amplitude é o jogo.
- Se o ensemble vencer no holdout: vira a régua do OD; senão, o próximo passo é treino ponderado por amplitude.
- Artefatos em `resultados/04b-ensemble-od/`: `metricas_baseline.csv`, `metricas_holdout.csv`, `modelos/lgbm_steps.pkl`, `modelos/dlinear_res_od.pt`, `modelos/ensemble.json`, `modelos/normalizacao.json` e `figs/`.
